# Appending Zarr v2 slices with time-series-friendly chunks

This notebook generates and appends 31 daily Zarr v2 slices in `inputs/zarr-v2`. New configurations use Zarr v3 by default, so these calls explicitly set `zarr_format=2`. The output is written below `outputs`, which is ignored by Git.

In [1]:
import os
import shutil
from pathlib import Path

import xarray as xr
from create_inputs import make_zarr_v2_slices

from zappend.api import zappend

## Generate daily input slices

The helper creates deterministic daily slices with dimensions `(time=1, lat=1800, lon=3600)`. Each day has a distinct constant value, which makes it easy to check that all inputs were appended in order.

In [2]:
%%time
make_zarr_v2_slices(31)

CPU times: user 5.25 s, sys: 986 ms, total: 6.24 s
Wall time: 4.36 s


## Collect the inputs

`zappend` accepts paths directly. Sorting the paths puts the daily slices in chronological order. The same list is used for both target layouts below.

In [3]:
input_dir = Path("inputs/zarr-v2")
output_dir =Path("outputs")
output_dir.mkdir(exist_ok=True)
slices = sorted(str(path) for path in input_dir.glob('*.zarr'))
slices[:3]

['inputs/zarr-v2/slice-2024-01-01.zarr',
 'inputs/zarr-v2/slice-2024-01-02.zarr',
 'inputs/zarr-v2/slice-2024-01-03.zarr']

## Default layout: favor spatial access

This first call keeps the input encoding. The source data use one day per time chunk and 450 by 450 spatial chunks, a sensible layout when maps or single days are the common access pattern. `force_new=True` makes the notebook repeatable by replacing only this example target.

In [4]:
%%time
full_time_chunk = len(slices)
target = f"{output_dir}/v2_example_space_opt.zarr"
count = zappend(
    slices,
    target_dir=target,
    zarr_format=2,
    force_new=True,
    logging="INFO",
)
assert count == full_time_chunk

2026-11-03 08:56:50,946 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-11-03 08:56:50,947 WARNING Permanently deleting outputs/v2_example_space_opt.zarr
2026-11-03 08:56:50,985 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-01.zarr
2026-11-03 08:56:50,999 INFO Creating target dataset outputs/v2_example_space_opt.zarr
2026-11-03 08:56:51,127 INFO Transaction completed.
2026-11-03 08:56:51,128 INFO Slice dataset inputs/zarr-v2/slice-2024-01-01.zarr closed
2026-11-03 08:56:51,137 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-02.zarr
2026-11-03 08:56:51,146 INFO Updating target dataset outputs/v2_example_space_opt.zarr
2026-11-03 08:56:51,291 INFO Transaction completed.
2026-11-03 08:56:51,292 INFO Slice dataset inputs/zarr-v2/slice-2024-01-02.zarr closed
2026-11-03 08:56:51,300 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-03.zarr
2026-11-03 08:56:51,311 INFO Updating target dataset outp

CPU times: user 6.5 s, sys: 1.2 s, total: 7.7 s
Wall time: 5.57 s


Opening the target shows its dimensions and the chunks inherited from the first slice. Compare this with the second target below.

In [5]:
%%time
ds = xr.open_zarr(target)
ds

CPU times: user 18.4 ms, sys: 4.25 ms, total: 22.6 ms
Wall time: 18.1 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

## Time-series layout

Here the data variables are written with one 31-day time chunk. A request for a point time series therefore reads one temporal chunk instead of 31 daily chunks. This trades more data read for map requests against fewer reads for time-series requests.

In [6]:
%%time
full_time_chunk = len(slices)
target = f"{output_dir}/v2_example_time_opt.zarr"
count = zappend(
    slices,
    target_dir=target,
    zarr_format=2,
    force_new=True,
    variables={
        'temperature': {'encoding': {'chunks': [full_time_chunk, 90, 90]}},
        'quality_flag': {'encoding': {'chunks': [full_time_chunk, 90, 90]}},
    },
    logging="INFO",
)
assert count == full_time_chunk

2026-11-03 08:56:56,598 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-11-03 08:56:56,599 WARNING Permanently deleting outputs/v2_example_time_opt.zarr
2026-11-03 08:56:56,646 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-01.zarr
2026-11-03 08:56:56,659 INFO Creating target dataset outputs/v2_example_time_opt.zarr
2026-11-03 08:56:58,421 INFO Transaction completed.
2026-11-03 08:56:58,421 INFO Slice dataset inputs/zarr-v2/slice-2024-01-01.zarr closed
2026-11-03 08:56:58,429 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-02.zarr
2026-11-03 08:56:58,436 INFO Updating target dataset outputs/v2_example_time_opt.zarr
2026-11-03 08:57:01,141 INFO Transaction completed.
2026-11-03 08:57:01,141 INFO Slice dataset inputs/zarr-v2/slice-2024-01-02.zarr closed
2026-11-03 08:57:01,150 INFO Opening slice dataset from inputs/zarr-v2/slice-2024-01-03.zarr
2026-11-03 08:57:01,158 INFO Updating target dataset outputs

CPU times: user 1min 55s, sys: 27.7 s, total: 2min 23s
Wall time: 1min 27s


In [7]:
%%time
ds = xr.open_zarr(target)
ds

CPU times: user 15.2 ms, sys: 4.39 ms, total: 19.6 ms
Wall time: 16.2 ms


<xarray.Dataset> Size: 1GB
Dimensions:       (time: 31, lat: 1800, lon: 3600)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * lat           (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon           (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.7 179.8 179.9
Data variables:
    quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
    temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
Attributes:
    title:    Zappend example slice

In [8]:
if os.path.exists(input_dir):
    shutil.rmtree(input_dir)